In [ ]:
import os
import shutil
import uuid

from pyspark.sql import functions as F
from pyspark.sql import types as T

In [ ]:
# ============================================================
# 0. Caminhos
# ============================================================

WORKSPACE_USER_PATH_LOCAL = "/Workspace/Users/ecleiton@gmail.com"

RAW_DATA_PATH_LOCAL = f"{WORKSPACE_USER_PATH_LOCAL}/raw"
PROCESSED_DATA_PATH_LOCAL = f"{WORKSPACE_USER_PATH_LOCAL}/processed"

RAW_DATA_PATH = f"file:{RAW_DATA_PATH_LOCAL}"

OFFERS_PATH = f"{RAW_DATA_PATH}/offers.json"
CUSTOMERS_PATH = f"{RAW_DATA_PATH}/customers.json"
TRANSACTIONS_PATH = f"{RAW_DATA_PATH}/transactions.json"

FINAL_PARQUET_PATH_LOCAL = (
    f"{PROCESSED_DATA_PATH_LOCAL}/customer_offer_dataset.parquet"
)

print("RAW_DATA_PATH_LOCAL:", RAW_DATA_PATH_LOCAL)
print("PROCESSED_DATA_PATH_LOCAL:", PROCESSED_DATA_PATH_LOCAL)
print("OFFERS_PATH:", OFFERS_PATH)
print("CUSTOMERS_PATH:", CUSTOMERS_PATH)
print("TRANSACTIONS_PATH:", TRANSACTIONS_PATH)
print("FINAL_PARQUET_PATH_LOCAL:", FINAL_PARQUET_PATH_LOCAL)

RAW_DATA_PATH_LOCAL: /Workspace/Users/ecleiton@gmail.com/raw
PROCESSED_DATA_PATH_LOCAL: /Workspace/Users/ecleiton@gmail.com/processed
OFFERS_PATH: file:/Workspace/Users/ecleiton@gmail.com/raw/offers.json
CUSTOMERS_PATH: file:/Workspace/Users/ecleiton@gmail.com/raw/customers.json
TRANSACTIONS_PATH: file:/Workspace/Users/ecleiton@gmail.com/raw/transactions.json
FINAL_PARQUET_PATH_LOCAL: /Workspace/Users/ecleiton@gmail.com/processed/customer_offer_dataset.parquet


In [ ]:
# ============================================================
# 1. Validação dos arquivos de entrada
# ============================================================

required_files = [
    "offers.json",
    "customers.json",
    "transactions.json",
]

if not os.path.exists(RAW_DATA_PATH_LOCAL):
    raise FileNotFoundError(f"Pasta raw não encontrada: {RAW_DATA_PATH_LOCAL}")

missing_files = [
    file_name
    for file_name in required_files
    if not os.path.exists(os.path.join(RAW_DATA_PATH_LOCAL, file_name))
]

if missing_files:
    raise FileNotFoundError(f"Arquivos ausentes na pasta raw: {missing_files}")

os.makedirs(PROCESSED_DATA_PATH_LOCAL, exist_ok=True)

print("Arquivos encontrados na raw:")
print(os.listdir(RAW_DATA_PATH_LOCAL))

Arquivos encontrados na raw:
['offers.json', 'customers.json', 'transactions.json']


In [ ]:
# ============================================================
# 2. Leitura dos JSONs
# ============================================================

df_offers = (
    spark.read
    .option("multiLine", True)
    .json(OFFERS_PATH)
    .withColumnRenamed("id", "offer_id")
)

df_customers = (
    spark.read
    .option("multiLine", True)
    .json(CUSTOMERS_PATH)
    .withColumnRenamed("id", "account_id")
)

df_transactions_raw = (
    spark.read
    .option("multiLine", True)
    .json(TRANSACTIONS_PATH)
)

df_transactions = (
    df_transactions_raw
    .withColumnRenamed("accountid", "account_id")
    .withColumnRenamed("time_since_test_start", "time")
)


In [ ]:
# ============================================================
# 3. Normalização do campo value
# ============================================================

value_schema = df_transactions.schema["value"].dataType

if not isinstance(value_schema, T.StructType):
    raise TypeError(
        "A coluna value não está como struct. Verifique a leitura do transactions.json."
    )

value_field_names = [field.name for field in value_schema.fields]

if "amount" in value_field_names:
    amount_col = F.col("value.amount").cast("double")
else:
    amount_col = F.lit(None).cast("double")

if "reward" in value_field_names:
    reward_col = F.col("value.reward").cast("double")
else:
    reward_col = F.lit(None).cast("double")

offer_id_candidates = []

if "offer_id" in value_field_names:
    offer_id_candidates.append(F.col("value.offer_id").cast("string"))

if "offer id" in value_field_names:
    offer_id_candidates.append(F.col("value.`offer id`").cast("string"))

if offer_id_candidates:
    offer_id_col = F.coalesce(*offer_id_candidates)
else:
    offer_id_col = F.lit(None).cast("string")

df_transactions = (
    df_transactions
    .withColumn("amount", amount_col)
    .withColumn("reward", reward_col)
    .withColumn("offer_id", offer_id_col)
    .drop("value")
)

In [ ]:
# ============================================================
# 4. Join transações + clientes + ofertas
# ============================================================

df_merged = (
    df_transactions
    .join(df_customers, on="account_id", how="left")
)

df_final = (
    df_merged
    .join(df_offers, on="offer_id", how="left")
)

In [ ]:
# ============================================================
# 5. Dummies dos canais
# ============================================================

df_final = (
    df_final
    .withColumn("channel_email", F.array_contains(F.col("channels"), "email").cast("int"))
    .withColumn("channel_mobile", F.array_contains(F.col("channels"), "mobile").cast("int"))
    .withColumn("channel_social", F.array_contains(F.col("channels"), "social").cast("int"))
    .withColumn("channel_web", F.array_contains(F.col("channels"), "web").cast("int"))
    .drop("channels")
)

channel_cols = [
    "channel_email",
    "channel_mobile",
    "channel_social",
    "channel_web",
]

for col_name in channel_cols:
    df_final = df_final.withColumn(
        col_name,
        F.coalesce(F.col(col_name), F.lit(0)).cast("int")
    )

In [ ]:
# ============================================================
# 6. Tratamento de clientes
# ============================================================

max_registered_on = (
    df_customers
    .select(F.max(F.to_date(F.col("registered_on").cast("string"), "yyyyMMdd")).alias("max_registered_on"))
    .collect()[0]["max_registered_on"]
)

df_final = (
    df_final
    .withColumn("gender", F.coalesce(F.col("gender"), F.lit("unknown")))
    .withColumn("age_missing_flag", (F.col("age") == F.lit(118)).cast("int"))
    .withColumn(
        "age",
        F.when(F.col("age") == F.lit(118), F.lit(None).cast("double"))
         .otherwise(F.col("age").cast("double"))
    )
    .withColumn(
        "credit_card_limit_missing_flag",
        F.col("credit_card_limit").isNull().cast("int")
    )
    .withColumn(
        "registered_on",
        F.to_date(F.col("registered_on").cast("string"), "yyyyMMdd")
    )
    .withColumn(
        "account_age_days",
        F.datediff(F.lit(max_registered_on), F.col("registered_on")).cast("int")
    )
)


In [ ]:
# ============================================================
# 7. Base de modelagem: uma linha por oferta recebida
# ============================================================

df_dataset = (
    df_final
    .filter(F.col("event") == "offer received")
    .withColumnRenamed("time", "received_time")
    .withColumn("offer_instance_id", F.monotonically_increasing_id())
    .withColumn("expiration_time", F.col("received_time") + F.col("duration"))
)

base_cols = [
    "offer_instance_id",
    "account_id",
    "offer_id",
    "received_time",
    "expiration_time",
]

In [ ]:
# ============================================================
# 8. Evento viewed dentro da janela da oferta
# ============================================================

df_viewed_events = (
    df_final
    .filter(F.col("event") == "offer viewed")
    .select(
        "account_id",
        "offer_id",
        F.col("time").alias("viewed_time")
    )
)

df_first_viewed = (
    df_dataset
    .select(*base_cols)
    .join(df_viewed_events, on=["account_id", "offer_id"], how="left")
    .filter(
        F.col("viewed_time").between(
            F.col("received_time"),
            F.col("expiration_time")
        )
    )
    .groupBy("offer_instance_id")
    .agg(F.min("viewed_time").alias("first_viewed_time"))
)

df_dataset = (
    df_dataset
    .join(df_first_viewed, on="offer_instance_id", how="left")
    .withColumn("viewed", F.col("first_viewed_time").isNotNull().cast("int"))
)

In [ ]:
# ============================================================
# 9. Evento completed dentro da janela da oferta
# ============================================================

df_completed_events = (
    df_final
    .filter(F.col("event") == "offer completed")
    .select(
        "account_id",
        "offer_id",
        F.col("time").alias("completed_time"),
        F.col("reward").alias("reward_completed")
    )
)

df_first_completed = (
    df_dataset
    .select(*base_cols)
    .join(df_completed_events, on=["account_id", "offer_id"], how="left")
    .filter(
        F.col("completed_time").between(
            F.col("received_time"),
            F.col("expiration_time")
        )
    )
    .groupBy("offer_instance_id")
    .agg(
        F.min("completed_time").alias("first_completed_time"),
        F.sum("reward_completed").alias("reward_completed")
    )
)

df_dataset = (
    df_dataset
    .join(df_first_completed, on="offer_instance_id", how="left")
    .withColumn("completed", F.col("first_completed_time").isNotNull().cast("int"))
    .withColumn("reward_completed", F.coalesce(F.col("reward_completed"), F.lit(0.0)))
)

In [ ]:
# ============================================================
# 10. Transações dentro da janela da oferta
# ============================================================

df_transaction_events = (
    df_final
    .filter(F.col("event") == "transaction")
    .select(
        "account_id",
        F.col("time").alias("transaction_time"),
        "amount"
    )
)

df_transactions_window = (
    df_dataset
    .select(*base_cols)
    .join(df_transaction_events, on="account_id", how="left")
    .filter(
        F.col("transaction_time").between(
            F.col("received_time"),
            F.col("expiration_time")
        )
    )
    .groupBy("offer_instance_id")
    .agg(
        F.count("amount").alias("transactions_during_offer"),
        F.sum("amount").alias("amount_during_offer"),
        F.avg("amount").alias("avg_ticket_during_offer"),
        F.min("transaction_time").alias("first_transaction_time"),
        F.max("transaction_time").alias("last_transaction_time")
    )
)

df_dataset = (
    df_dataset
    .join(df_transactions_window, on="offer_instance_id", how="left")
    .withColumn(
        "transactions_during_offer",
        F.coalesce(F.col("transactions_during_offer"), F.lit(0)).cast("int")
    )
    .withColumn(
        "amount_during_offer",
        F.coalesce(F.col("amount_during_offer"), F.lit(0.0))
    )
    .withColumn(
        "avg_ticket_during_offer",
        F.coalesce(F.col("avg_ticket_during_offer"), F.lit(0.0))
    )
)

In [ ]:
# ============================================================
# 11. Remoção de colunas de evento original
# ============================================================

cols_to_drop = [
    "event",
    "amount",
    "reward",
]

df_dataset = df_dataset.drop(*[col for col in cols_to_drop if col in df_dataset.columns])

In [ ]:
# ============================================================
# 12. Features finais
# ============================================================

df_dataset = (
    df_dataset
    .withColumn("registered_year", F.year("registered_on").cast("int"))
    .withColumn("registered_month", F.month("registered_on").cast("int"))
    .withColumn(
        "time_to_view",
        (F.col("first_viewed_time") - F.col("received_time")).cast("double")
    )
    .withColumn(
        "time_to_complete",
        (F.col("first_completed_time") - F.col("received_time")).cast("double")
    )
    .withColumn(
        "time_to_first_transaction",
        (F.col("first_transaction_time") - F.col("received_time")).cast("double")
    )
    .withColumn(
        "time_to_last_transaction",
        (F.col("last_transaction_time") - F.col("received_time")).cast("double")
    )
    .withColumn(
        "has_transaction_during_offer",
        (F.col("transactions_during_offer") > 0).cast("int")
    )
    .withColumn(
        "has_reward",
        (F.col("reward_completed") > 0).cast("int")
    )
    .withColumn(
        "viewed_but_not_completed",
        ((F.col("viewed") == 1) & (F.col("completed") == 0)).cast("int")
    )
    .withColumn(
        "purchased_without_viewing",
        ((F.col("viewed") == 0) & (F.col("has_transaction_during_offer") == 1)).cast("int")
    )
    .withColumn(
        "purchased_without_reward",
        ((F.col("has_transaction_during_offer") == 1) & (F.col("has_reward") == 0)).cast("int")
    )
    .withColumn(
        "net_amount_after_reward",
        F.col("amount_during_offer") - F.col("reward_completed")
    )
    .withColumn(
        "reward_rate_over_amount",
        F.when(
            F.col("amount_during_offer") > 0,
            F.col("reward_completed") / F.col("amount_during_offer")
        ).otherwise(F.lit(0.0))
    )
    .withColumn("is_bogo", (F.col("offer_type") == "bogo").cast("int"))
    .withColumn("is_discount", (F.col("offer_type") == "discount").cast("int"))
    .withColumn("is_informational", (F.col("offer_type") == "informational").cast("int"))
    .withColumn(
        "is_no_discount_offer",
        (
            (F.col("offer_type") == "informational") |
            (F.coalesce(F.col("discount_value"), F.lit(0.0)) == 0)
        ).cast("int")
    )
)


In [ ]:
# ============================================================
# 13. Garantia de tipos
# ============================================================

numeric_cols = [
    "received_time",
    "age",
    "credit_card_limit",
    "min_value",
    "duration",
    "discount_value",
    "account_age_days",
    "expiration_time",
    "first_viewed_time",
    "first_completed_time",
    "reward_completed",
    "transactions_during_offer",
    "amount_during_offer",
    "avg_ticket_during_offer",
    "first_transaction_time",
    "last_transaction_time",
    "registered_year",
    "registered_month",
    "time_to_view",
    "time_to_complete",
    "time_to_first_transaction",
    "time_to_last_transaction",
    "net_amount_after_reward",
    "reward_rate_over_amount",
]

for col_name in numeric_cols:
    if col_name in df_dataset.columns:
        df_dataset = df_dataset.withColumn(col_name, F.col(col_name).cast("double"))

flag_cols = [
    "age_missing_flag",
    "credit_card_limit_missing_flag",
    "viewed",
    "completed",
    "has_transaction_during_offer",
    "has_reward",
    "viewed_but_not_completed",
    "purchased_without_viewing",
    "purchased_without_reward",
    "is_bogo",
    "is_discount",
    "is_informational",
    "is_no_discount_offer",
    "channel_email",
    "channel_mobile",
    "channel_social",
    "channel_web",
]

for col_name in flag_cols:
    if col_name in df_dataset.columns:
        df_dataset = df_dataset.withColumn(
            col_name,
            F.coalesce(F.col(col_name), F.lit(0)).cast("int")
        )

count_cols = [
    "offer_instance_id",
    "transactions_during_offer",
    "registered_year",
    "registered_month",
    "account_age_days",
]

for col_name in count_cols:
    if col_name in df_dataset.columns:
        df_dataset = df_dataset.withColumn(col_name, F.col(col_name).cast("long"))

cat_cols = [
    "account_id",
    "offer_id",
    "gender",
    "offer_type",
]

for col_name in cat_cols:
    if col_name in df_dataset.columns:
        df_dataset = df_dataset.withColumn(col_name, F.col(col_name).cast("string"))

In [ ]:
# ============================================================
# 14. Ordenação final das colunas
# ============================================================

ordered_cols = [
    "offer_instance_id",
    "account_id",
    "offer_id",

    "received_time",
    "expiration_time",
    "duration",

    "age",
    "age_missing_flag",
    "gender",
    "registered_on",
    "registered_year",
    "registered_month",
    "account_age_days",
    "credit_card_limit",
    "credit_card_limit_missing_flag",

    "offer_type",
    "min_value",
    "discount_value",
    "is_bogo",
    "is_discount",
    "is_informational",
    "is_no_discount_offer",

    "channel_email",
    "channel_mobile",
    "channel_social",
    "channel_web",

    "first_viewed_time",
    "viewed",
    "time_to_view",

    "first_completed_time",
    "completed",
    "time_to_complete",
    "reward_completed",
    "has_reward",

    "transactions_during_offer",
    "has_transaction_during_offer",
    "amount_during_offer",
    "avg_ticket_during_offer",
    "first_transaction_time",
    "last_transaction_time",
    "time_to_first_transaction",
    "time_to_last_transaction",

    "viewed_but_not_completed",
    "purchased_without_viewing",
    "purchased_without_reward",

    "net_amount_after_reward",
    "reward_rate_over_amount",
]

ordered_cols = [col for col in ordered_cols if col in df_dataset.columns]
remaining_cols = [col for col in df_dataset.columns if col not in ordered_cols]

df_dataset = df_dataset.select(ordered_cols + remaining_cols)

In [ ]:
# ============================================================
# 15. Validações estruturais
# ============================================================

checks = {
    "rows": df_dataset.count(),
    "unique_offer_instance_id": df_dataset.select("offer_instance_id").distinct().count(),
    "duplicated_offer_instance_id": (
        df_dataset.count()
        - df_dataset.select("offer_instance_id").distinct().count()
    ),
    "missing_account_id": df_dataset.filter(F.col("account_id").isNull()).count(),
    "missing_offer_id": df_dataset.filter(F.col("offer_id").isNull()).count(),
    "view_time_before_received": df_dataset.filter(
        F.col("first_viewed_time").isNotNull()
        & (F.col("first_viewed_time") < F.col("received_time"))
    ).count(),
    "view_time_after_expiration": df_dataset.filter(
        F.col("first_viewed_time").isNotNull()
        & (F.col("first_viewed_time") > F.col("expiration_time"))
    ).count(),
    "complete_time_before_received": df_dataset.filter(
        F.col("first_completed_time").isNotNull()
        & (F.col("first_completed_time") < F.col("received_time"))
    ).count(),
    "complete_time_after_expiration": df_dataset.filter(
        F.col("first_completed_time").isNotNull()
        & (F.col("first_completed_time") > F.col("expiration_time"))
    ).count(),
    "transaction_time_before_received": df_dataset.filter(
        F.col("first_transaction_time").isNotNull()
        & (F.col("first_transaction_time") < F.col("received_time"))
    ).count(),
    "transaction_time_after_expiration": df_dataset.filter(
        F.col("last_transaction_time").isNotNull()
        & (F.col("last_transaction_time") > F.col("expiration_time"))
    ).count(),
    "reward_without_completed": df_dataset.filter(
        (F.col("completed") == 0)
        & (F.col("reward_completed") > 0)
    ).count(),
    "completed_without_reward_non_informational": df_dataset.filter(
        (F.col("completed") == 1)
        & (F.col("reward_completed") <= 0)
        & (F.col("offer_type") != "informational")
    ).count(),
}

df_checks = spark.createDataFrame(
    [(key, int(value)) for key, value in checks.items()],
    ["check", "value"]
)

display(df_checks)

check,value
rows,76277
unique_offer_instance_id,76277
duplicated_offer_instance_id,0
missing_account_id,0
missing_offer_id,0
view_time_before_received,0
view_time_after_expiration,0
complete_time_before_received,0
complete_time_after_expiration,0
transaction_time_before_received,0


In [ ]:
# ============================================================
# 16. Relatório de nulos
# ============================================================

total_rows = checks["rows"]

null_exprs = [
    F.sum(F.col(col_name).isNull().cast("int")).alias(col_name)
    for col_name in df_dataset.columns
]

df_null_summary = (
    df_dataset
    .agg(*null_exprs)
    .select(
        F.explode(
            F.array(
                *[
                    F.struct(
                        F.lit(col_name).alias("column"),
                        F.col(col_name).cast("long").alias("missing_count")
                    )
                    for col_name in df_dataset.columns
                ]
            )
        ).alias("x")
    )
    .select("x.*")
    .withColumn(
        "missing_rate",
        F.when(F.lit(total_rows) > 0, F.col("missing_count") / F.lit(total_rows))
         .otherwise(F.lit(0.0))
    )
    .orderBy(F.desc("missing_count"))
)

display(df_null_summary)

column,missing_count,missing_rate
time_to_complete,42646,0.559093829070362
first_completed_time,42646,0.559093829070362
time_to_view,19382,0.25410018747459917
first_viewed_time,19382,0.25410018747459917
first_transaction_time,16325,0.21402257561257
time_to_first_transaction,16325,0.21402257561257
last_transaction_time,16325,0.21402257561257
time_to_last_transaction,16325,0.21402257561257
age,9776,0.12816445324278616
credit_card_limit,9776,0.12816445324278616


In [ ]:
# ============================================================
# 17. Resumo por tipo de oferta
# ============================================================

df_summary_offer_type = (
    df_dataset
    .groupBy("offer_type")
    .agg(
        F.count("offer_instance_id").alias("qtd_ofertas_recebidas"),
        F.countDistinct("account_id").alias("clientes_unicos"),
        F.avg("viewed").alias("taxa_visualizacao"),
        F.avg("completed").alias("taxa_conclusao"),
        F.avg("has_transaction_during_offer").alias("taxa_compra_durante_oferta"),
        F.sum("amount_during_offer").alias("valor_total"),
        F.sum("reward_completed").alias("recompensa_total"),
        F.sum("net_amount_after_reward").alias("valor_liquido"),
        F.avg("avg_ticket_during_offer").alias("ticket_medio")
    )
    .orderBy("offer_type")
)

display(df_summary_offer_type)

offer_type,qtd_ofertas_recebidas,clientes_unicos,taxa_visualizacao,taxa_conclusao,taxa_compra_durante_oferta,valor_total,recompensa_total,valor_liquido,ticket_medio
bogo,30499,14992,0.8318961277418931,0.5137545493294862,0.8040263615200498,808643.3199999962,117920.0,690723.3199999975,10.90578019711266
discount,30543,14945,0.705857315915267,0.5880889238123301,0.8495236224339456,1068233.2000000076,54731.0,1013502.2000000062,11.645421425994941
informational,15235,10547,0.6540203478831638,0.0,0.6224483098129308,210286.7199999995,0.0,210286.7199999995,8.216350922843692


In [ ]:
# ============================================================
# 18. Resumo por oferta
# ============================================================

group_offer_cols = [
    "offer_id",
    "offer_type",
    "min_value",
    "duration",
    "discount_value",
    "channel_email",
    "channel_mobile",
    "channel_social",
    "channel_web",
]

df_summary_offer = (
    df_dataset
    .groupBy(*group_offer_cols)
    .agg(
        F.count("offer_instance_id").alias("qtd_ofertas_recebidas"),
        F.countDistinct("account_id").alias("clientes_unicos"),
        F.avg("viewed").alias("taxa_visualizacao"),
        F.avg("completed").alias("taxa_conclusao"),
        F.avg("has_transaction_during_offer").alias("taxa_compra_durante_oferta"),
        F.sum("amount_during_offer").alias("valor_total"),
        F.sum("reward_completed").alias("recompensa_total"),
        F.sum("net_amount_after_reward").alias("valor_liquido"),
        F.avg("avg_ticket_during_offer").alias("ticket_medio")
    )
    .orderBy(F.desc("qtd_ofertas_recebidas"))
)

display(df_summary_offer)

offer_id,offer_type,min_value,duration,discount_value,channel_email,channel_mobile,channel_social,channel_web,qtd_ofertas_recebidas,clientes_unicos,taxa_visualizacao,taxa_conclusao,taxa_compra_durante_oferta,valor_total,recompensa_total,valor_liquido,ticket_medio
9b98b8c7a33c4b65b9aebfe6a799e6d9,bogo,5.0,7.0,5.0,1,1,0,1,7677,6355,0.5442230037775173,0.5671486257652729,0.7880682558290999,196912.04,22770.0,174142.03999999995,10.563691948467344
0b1e1539f2cc45b7b9fa7c272da2e1d7,discount,20.0,10.0,5.0,1,0,0,1,7668,6374,0.35628586332811685,0.44861763171622326,0.8485915492957746,253377.34000000005,18605.0,234772.34000000003,11.87454382296832
ae264e3637204a6fb9bb56bc8210ddfd,bogo,10.0,7.0,10.0,1,1,1,0,7658,6374,0.8772525463567511,0.48158788195351265,0.8469574301384173,239701.61000000004,38630.0,201071.61000000004,11.303024818964587
2298d6c36e964ae4a3e7e9706d1fb8c2,discount,7.0,7.0,3.0,1,1,1,1,7646,6325,0.961679309442846,0.6755166099921528,0.85731101229401,251993.78,16152.0,235841.77999999997,11.332604140987184
2906b810c7d4411798c6938adc9daaa5,discount,10.0,7.0,2.0,1,1,0,1,7632,6285,0.5397012578616353,0.5273846960167715,0.7877358490566038,202780.44999999992,8446.0,194334.44999999998,10.957854376820771
5a8bc65990b245e5a138643cd4eb9837,informational,0.0,3.0,0.0,1,1,1,0,7618,6320,0.8075610396429509,0.0,0.6223418220005251,101131.87999999996,0.0,101131.87999999996,8.100474709022492
3f207df678b143eea3cee63160fa8bed,informational,0.0,4.0,0.0,1,1,0,1,7617,6331,0.5004594984902192,0.0,0.622554811605619,109154.84000000005,0.0,109154.84000000005,8.33224234950643
fafdcd668e3743c1bb461111dcafc2a4,discount,10.0,10.0,2.0,1,1,1,1,7597,6332,0.9681453205212583,0.701855995787811,0.9046992233776491,360081.62999999995,11528.0,348553.6299999999,12.419727354713494
4d5c57ea9a6940dd891ad53e9dbe8da0,bogo,10.0,5.0,10.0,1,1,1,1,7593,6330,0.954563413670486,0.4386935335177137,0.7921770051363097,187736.61000000002,34370.0,153366.61,10.93142996055264
f19421c1d4aa40978ebb69ca19b0e20d,bogo,5.0,5.0,5.0,1,1,1,1,7571,6262,0.9546955488046494,0.5674283450006604,0.7886672830537578,184293.05999999994,22150.0,162143.06,10.825124209384434


In [ ]:

# ============================================================
# 19. Salvamento da base.
# Observação:
    # Foi necessário converter para pandas pois não estava permitindo salvar o parquet.
    # por pyspark direto e com a conversão funcionou.
# ============================================================
import os
import pandas as pd

PROCESSED_DATA_PATH_LOCAL = "/Workspace/Users/ecleiton@gmail.com/processed"

os.makedirs(PROCESSED_DATA_PATH_LOCAL, exist_ok=True)

PARQUET_OUTPUT_PATH_LOCAL = os.path.join(
    PROCESSED_DATA_PATH_LOCAL,
    "customer_offer_dataset.parquet"
)

# Coleta os dados do Spark para o driver sem usar toPandas()
rows = df_dataset.collect()
columns = df_dataset.columns

df_dataset_pd = pd.DataFrame(
    [row.asDict(recursive=True) for row in rows],
    columns=columns
)

df_dataset_pd.to_parquet(
    PARQUET_OUTPUT_PATH_LOCAL,
    index=False
)

print("Base final salva com sucesso.")
print("Parquet:", PARQUET_OUTPUT_PATH_LOCAL)
print("Shape final:", df_dataset_pd.shape)

Base final salva com sucesso.
Parquet: /Workspace/Users/ecleiton@gmail.com/processed/customer_offer_dataset.parquet
Shape final: (76277, 47)
